In [9]:
%load_ext autoreload
%autoreload 2
import sys, os

# 获取项目根目录：当前笔记本文件的目录再上一级
project_root = os.path.abspath('..')   # 如果笔记本在 notebook/ 下，否则改为 os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"项目根目录: {project_root}")

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
项目根目录: /data1/yesiliang/rag


In [ ]:
# 写一个快速排序函数

In [7]:
#load_longbench_dataset_with_chunking()

# load_longbench_with_refined_chunking()


#chunks = split_chunks_with_cleaning(file_path)

#print(f"清洗并分块完成，共提取出 {len(chunks)} 条高质量文档块。")

# 预览一下清洗后的效果
# for i, chunk in enumerate(chunks[:3], 1):
#     print(f"--- 经清洗后的块 {i} ---")
#     print(f"{chunk[:200]}") # 看看是否还有 <br>
#     print()

#update_eval_file('longbench_eval.json', 'longbench_eval_with_gt.json')

# --- 执行示例 ---
# 1. 运行你之前的 generate_contextual_chunks 得到列表
#enhanced_chunks = generate_contextual_chunks('longbench_kb.md')

# 2. 保存为原样格式的文件
#save_enhanced_kb_to_disk(enhanced_chunks, "longbench_kb_contextual.md")

#save_embeddings(real_ids, chunks, embeddings)
#print(real_ids[0],embeddings[0])
#client.delete_collection("baai_bge_collection")
#client.delete_collection("contextual_collection")

🚀 正在启动：智能清洗 + 递归分块模式...
✅ 构建完成！知识库块数: 481，测试题数: 50
📊 样例：第一题被切分为 14 个语义连贯的知识块。


In [10]:

#real_ids, chunks = parse_kb_file('dureader_kb.md')
#embeddings = embedding_model.encode(chunks, normalize_embeddings=True).tolist()
#save_embeddings(real_ids, chunks, embeddings)

成功存入 1263 条向量数据。


In [26]:
import sys
sys.path.append('/data1/yesiliang/rag/')
import pickle
import numpy as np
from src.rerank_utils import colbert_model, parse_kb_file  # 确保 colbert_model 已加载

# ---------- 配置 ----------
KB_FILE = "/data1/yesiliang/rag/data/dureader/dureader_kb.md"          # 新知识库文件
CACHE_FILE = "/data1/yesiliang/rag/vector_db/colbert_cache_dureader.pkl"

# ---------- 1. 解析知识库 ----------
print(f"🔍 解析知识库: {KB_FILE}")
real_ids, real_chunks = parse_kb_file(KB_FILE)
print(f"   共解析出 {len(real_ids)} 个文本块")

# ---------- 2. 批量计算 ColBERT 多向量 ----------
print(f"🚀 开始批量计算 ColBERT 向量（共 {len(real_chunks)} 个块）...")
outputs = colbert_model.encode(real_chunks, return_colbert_vecs=True)
colbert_vecs = outputs['colbert_vecs']  # list of numpy arrays, 每个 [Ld, D]

# ---------- 3. 建立 ID -> 向量的映射 ----------
print("📦 建立 ID -> 向量缓存...")
cache = {}
for chunk_id, vec in zip(real_ids, colbert_vecs):
    cache[chunk_id] = np.asarray(vec)  # 转成 numpy 数组，方便后续使用

# ---------- 4. 保存到磁盘 ----------
with open(CACHE_FILE, 'wb') as f:
    pickle.dump(cache, f)

print(f"✅ 预计算完成！缓存已保存至: {CACHE_FILE} (共 {len(cache)} 个向量)")

🔍 解析知识库: /data1/yesiliang/rag/data/dureader/dureader_kb.md
   共解析出 1263 个文本块
🚀 开始批量计算 ColBERT 向量（共 1263 个块）...


pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 27.25it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 5/5 [00:04<00:00,  1.12it/s]


📦 建立 ID -> 向量缓存...
✅ 预计算完成！缓存已保存至: /data1/yesiliang/rag/vector_db/colbert_cache_dureader.pkl (共 1263 个向量)


In [1]:
import sys
print(sys.executable)

/data1/yesiliang/rag/.venv/bin/python


In [9]:
import sys
sys.path.append('/data1/yesiliang/rag/')
# 小测试：验证问题拆解函数
from src.rerank_utils import decompose_question

test_question = "南通海安是不是很穷"

print(f"原始问题: {test_question}\n")
subs = decompose_question(test_question)
print(f"拆解出 {len(subs)} 个子问题:")
for i, sq in enumerate(subs, 1):
    print(f"  {i}. {sq}")

原始问题: 南通海安是不是很穷

拆解出 4 个子问题:
  1. 海安市（原海安县）是否被列为国家级或省级贫困县？
  2. 海安市的人均GDP是多少？
  3. 海安市的城乡居民人均可支配收入是多少？
  4. 南通市所辖各区县的经济总量排名中，海安市位于什么位置？


In [ ]:
import json
import time
import sys
import torch
import gc
sys.path.append('/data1/yesiliang/rag/')
from src.rerank_utils import get_chunks_by_question_id, decompose_question, bge_pointwise_rerank, generate

# 我只对前 3 个问题做循环，不打答案，只看显存
with open("dureader_eval.json", "r", encoding="utf-8") as f:
    test_list = json.load(f)["test"][:3]

def print_gpu_mem(step_label):
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1024**3
        print(f"   [{step_label}] 显存已分配: {mem:.2f} GB")

for idx, qa in enumerate(test_list):
    q = qa["question"]
    print(f"\n==== 处理第 {idx} 题 ====")
    print_gpu_mem("BEFORE chunk load")   # 查看当前显存

    chunks = get_chunks_by_question_id(idx)
    print_gpu_mem("AFTER chunk load")   # 获取题目文本块后的显存

    subs = decompose_question(q)        # 调用你的拆解函数
    print_gpu_mem("AFTER decompose")    # 拆解后的显存

    ids = [f"题_{idx}_块_{j}" for j in range(len(chunks))]
    for _k, sub_q in enumerate(subs):
        _ = bge_pointwise_rerank(sub_q, ids, chunks)
    print_gpu_mem("AFTER bge rerank")   # 进行重排后的显存

    # 没用 del 和 empty_cache 时，显存理论上会一点点累积上去
    # 如果这里打印出的数字一直上升，就是 OOM 了。

    time.sleep(1)  # 稍微喘口气，方便你看日志

🧪 测试问题: 热诚传说结局

🟢 开始: 1. 获取该题所有块
   块数: 25
🔴 完成: 1. 获取该题所有块 | 耗时: 0.01 秒
🟢 开始: 2. 问题拆解
   子问题: 3 个
🔴 完成: 2. 问题拆解 | 耗时: 2.11 秒
🟢 开始: 3. 逐个子问题进行 Pointwise 检索
🟢 开始:    3.1 子问题检索: 热诚传说的结局是什么？
      选中 3 个不重复块
🔴 完成:    3.1 子问题检索: 热诚传说的结局是什么？ | 耗时: 0.18 秒
🟢 开始:    3.2 子问题检索: 热诚传说是否有多个结局？
      选中 3 个不重复块
🔴 完成:    3.2 子问题检索: 热诚传说是否有多个结局？ | 耗时: 0.15 秒
🟢 开始:    3.3 子问题检索: 热诚传说结局的详细情节是怎样的？
      选中 3 个不重复块
🔴 完成:    3.3 子问题检索: 热诚传说结局的详细情节是怎样的？ | 耗时: 0.15 秒
🔴 完成: 3. 逐个子问题进行 Pointwise 检索 | 耗时: 0.48 秒
🟢 开始: 4. DeepSeek 生成答案
   答案: 《热诚传说》结局中，史雷成为新的导师，醒来时已是几百年后（或几年后），城市现代化，公主早已去世；官方剧情被指为百合结局。...
🔴 完成: 4. DeepSeek 生成答案 | 耗时: 3.65 秒

✅ 单步测试完毕


In [ ]:
from src.generate import (
    evaluate_rag_system,
    strategy_pairwise,
    strategy_listwise,
    strategy_colbert,
    strategy_pointwise,
    strategy_baseline,
    strategy_colbert_listwise,
    strategy_setselection,
    # strategy_listwise_setselection,
    # strategy_pointwise_dureader
)
import json

# 1. 加载数据
with open("/data1/yesiliang/rag/data/mutifieldqa/mutifieldqa_eval.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_questions = test_data["test"]

# 2. 【核心修改点】在这里选择你想使用的策略函数名
#CURRENT_STRATEGY = strategy_baseline
#CURRENT_STRATEGY = strategy_pointwise
CURRENT_STRATEGY = strategy_listwise
#CURRENT_STRATEGY = strategy_pairwise
#CURRENT_STRATEGY = strategy_colbert 
#CURRENT_STRATEGY = strategy_colbert_listwise 
#CURRENT_STRATEGY = strategy_setselection
#CURRENT_STRATEGY = strategy_listwise_setselection
#CURRENT_STRATEGY = strategy_pointwise_dureader
# 3. 设置对应的输出文件名
output_mapping = {
    strategy_baseline: "multifieldqa_baseline.json",
    strategy_pointwise: "multifieldqa_pointwise.json",
    strategy_colbert: "multifieldqa_colbert.json",
    strategy_listwise: "multifieldqa_listwise.json",
    strategy_pairwise: "multifieldqa_pairwise.json",
    strategy_colbert_listwise: "bge_colbert_deepseek_listwise.json",
    strategy_setselection:"dureader_setselection.json",
    # strategy_listwise_setselection: "dureader_listwise_setselection.json",
    # strategy_pointwise_dureader: "dureader_pointwise.json"
}

output_file = output_mapping[CURRENT_STRATEGY]
output_path = f"/data1/yesiliang/rag/results/{output_file}" 
# 4. 执行并保存
test_results = evaluate_rag_system(test_questions, CURRENT_STRATEGY, top_k_for_gen=2, skip_retrieval=False)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_results, f, indent=2, ensure_ascii=False)

print(f"评估完成！使用策略: {CURRENT_STRATEGY.__name__}, 结果存至: {output_path}")

开始评估，策略名称: strategy_pointwise，总计 50 个问题...


100%|██████████| 50/50 [03:28<00:00,  4.18s/it]

评估完成！使用策略: strategy_pointwise, 结果存至: /data1/yesiliang/rag/results/multifieldqa_pointwise.json


In [28]:
from src.generate import calculate_metrics

GT_FILE = '/data1/yesiliang/rag/data/mutifieldqa/mutifieldqa_eval.json'
PRED_FILE = '/data1/yesiliang/rag/results/multifieldqa_pointwise.json' 
    
calculate_metrics(GT_FILE, PRED_FILE)


检索效果评估报告
----------------------------------------
评估题目总数:  50
Hit Rate @ 1:  0.4200
Hit Rate @ 2:  0.5200
Hit Rate @ 20: 0.9200
MRR:           0.5637


In [29]:
from src.generate import run_gen_evaluation

run_gen_evaluation("/data1/yesiliang/rag/results/multifieldqa_pointwise.json")


生成质量评估报告: /data1/yesiliang/rag/results/multifieldqa_pointwise.json
---------------------------------------------
评估样本数: 50
Exact Match (EM): 0.3000
F1 Score:         0.5344
Precision (P):    0.5861
Recall (R):       0.5607
